# Reproducing [MIMIC_IV_MEDS](https://github.com/Medical-Event-Data-Standard/MIMIC_IV_MEDS/blob/main/src/MIMIC_IV_MEDS/configs/ETL.yaml)
Setting some Paths

In [1]:
from pathlib import Path
import shutil
import subprocess
import os

# Hard coded paths
RAW_SOURCE_ROOT = Path("/workspaces/example/data/physionet.org/files/mimic-demo/2.2")
NOTEBOOK_ROOT = Path("/workspaces/OpenICU.example/output/project/mimiciv-meds-notebook-run")

RAW_INPUT_DIR = NOTEBOOK_ROOT / "raw_input"
PRE_MEDS_DIR = NOTEBOOK_ROOT / "pre_MEDS"
MEDS_COHORT_DIR = NOTEBOOK_ROOT / "MEDS_cohort"

print("RAW_SOURCE_ROOT =", RAW_SOURCE_ROOT)
print("NOTEBOOK_ROOT   =", NOTEBOOK_ROOT)

RAW_SOURCE_ROOT = /workspaces/example/data/physionet.org/files/mimic-demo/2.2
NOTEBOOK_ROOT   = /workspaces/OpenICU.example/output/project/mimiciv-meds-notebook-run


## Placing the raw [mimic demo data](https://physionet.org/content/mimic-iv-demo/2.2/) into our project folder *NOTEBOOK_ROOT*

In [2]:
if NOTEBOOK_ROOT.exists():
    shutil.rmtree(NOTEBOOK_ROOT)

NOTEBOOK_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copytree(RAW_SOURCE_ROOT, RAW_INPUT_DIR)

print("Copied raw input to:", RAW_INPUT_DIR)

Copied raw input to: /workspaces/OpenICU.example/output/project/mimiciv-meds-notebook-run/raw_input


## Prints

In [3]:
from MIMIC_IV_MEDS import ETL_CFG, EVENT_CFG, RUNNER_CFG, dataset_info, HAS_PRE_MEDS  # ty:ignore[unresolved-import]

print("ETL_CFG     =", ETL_CFG)
print("EVENT_CFG   =", EVENT_CFG)
print("RUNNER_CFG  =", RUNNER_CFG)
print("HAS_PRE_MEDS =", HAS_PRE_MEDS)
print("dataset_name =", dataset_info.dataset_name)
print("dataset_version =", dataset_info.raw_dataset_version)

ETL_CFG     = /workspaces/.test/lib/python3.13/site-packages/MIMIC_IV_MEDS/configs/ETL.yaml
EVENT_CFG   = /workspaces/.test/lib/python3.13/site-packages/MIMIC_IV_MEDS/configs/event_configs.yaml
RUNNER_CFG  = /workspaces/.test/lib/python3.13/site-packages/MIMIC_IV_MEDS/configs/runner.yaml
HAS_PRE_MEDS = True
dataset_name = MIMIC-IV
dataset_version = 3.1


## PreMedsTransform

If available configs, tranform raw data, else use raw data. That is the pre meds data

In [4]:
from MIMIC_IV_MEDS.pre_MEDS import main as pre_MEDS_transform  # ty:ignore[unresolved-import]

pre_MEDS_transform(
    input_dir=RAW_INPUT_DIR,
    output_dir=PRE_MEDS_DIR,
    do_overwrite=False,
    do_copy=True,
)

print("pre_MEDS finished.")
print("PRE_MEDS_DIR =", PRE_MEDS_DIR)

pre_MEDS finished.
PRE_MEDS_DIR = /workspaces/OpenICU.example/output/project/mimiciv-meds-notebook-run/pre_MEDS


## Print pre_meds

In [5]:
for p in sorted(PRE_MEDS_DIR.rglob("*")):
    print(p)

/workspaces/OpenICU.example/output/project/mimiciv-meds-notebook-run/pre_MEDS/.done
/workspaces/OpenICU.example/output/project/mimiciv-meds-notebook-run/pre_MEDS/demo_subject_id.csv
/workspaces/OpenICU.example/output/project/mimiciv-meds-notebook-run/pre_MEDS/hosp
/workspaces/OpenICU.example/output/project/mimiciv-meds-notebook-run/pre_MEDS/hosp/admissions.csv.gz
/workspaces/OpenICU.example/output/project/mimiciv-meds-notebook-run/pre_MEDS/hosp/d_hcpcs.csv.gz
/workspaces/OpenICU.example/output/project/mimiciv-meds-notebook-run/pre_MEDS/hosp/d_icd_diagnoses.parquet
/workspaces/OpenICU.example/output/project/mimiciv-meds-notebook-run/pre_MEDS/hosp/d_icd_procedures.parquet
/workspaces/OpenICU.example/output/project/mimiciv-meds-notebook-run/pre_MEDS/hosp/d_labitems.csv.gz
/workspaces/OpenICU.example/output/project/mimiciv-meds-notebook-run/pre_MEDS/hosp/diagnoses_icd.parquet
/workspaces/OpenICU.example/output/project/mimiciv-meds-notebook-run/pre_MEDS/hosp/drgcodes.parquet
/workspaces/Ope

## Wrapper for running CLI commands with logging

In [6]:
def run_command(cmd, env=None, cwd=None, print_cmd=True):
    if print_cmd:
        print("RUNNING:")
        print(" ".join(map(str, cmd)))
        print()

    result = subprocess.run(
        cmd,
        text=True,
        capture_output=True,
        env=env,
        cwd=cwd,
    )

    print("RETURN CODE:", result.returncode)
    print("\nSTDOUT:\n")
    print(result.stdout)
    print("\nSTDERR:\n")
    print(result.stderr)

    return result

## Initialize environment variables for running the MEDS transformation pipeline.

In [7]:
from MIMIC_IV_MEDS import __version__ as PKG_VERSION  # ty:ignore[unresolved-import]

base_env = os.environ.copy()
base_env["HYDRA_FULL_ERROR"] = "1"

base_env["DATASET_NAME"] = dataset_info.dataset_name
base_env["DATASET_VERSION"] = f"{dataset_info.raw_dataset_version}:{PKG_VERSION}"
base_env["EVENT_CONVERSION_CONFIG_FP"] = str(EVENT_CFG.resolve())
base_env["PRE_MEDS_DIR"] = str(PRE_MEDS_DIR.resolve())
base_env["MEDS_COHORT_DIR"] = str(MEDS_COHORT_DIR.resolve())

for k in [
    "DATASET_NAME",
    "DATASET_VERSION",
    "EVENT_CONVERSION_CONFIG_FP",
    "PRE_MEDS_DIR",
    "MEDS_COHORT_DIR",
]:
    print(k, "=", base_env[k])

DATASET_NAME = MIMIC-IV
DATASET_VERSION = 3.1:0.0.7
EVENT_CONVERSION_CONFIG_FP = /workspaces/.test/lib/python3.13/site-packages/MIMIC_IV_MEDS/configs/event_configs.yaml
PRE_MEDS_DIR = /workspaces/OpenICU.example/output/project/mimiciv-meds-notebook-run/pre_MEDS
MEDS_COHORT_DIR = /workspaces/OpenICU.example/output/project/mimiciv-meds-notebook-run/MEDS_cohort


# Executing all etl stages

In [8]:
# cmd = [
#     "MEDS_transform-runner",
#     f"--config-path={str(RUNNER_CFG.parent.resolve())}",
#     f"--config-name={RUNNER_CFG.stem}",
#     f"pipeline_config_fp={str(ETL_CFG.resolve())}",
#     "~parallelize",
#     "hydra.searchpath=[pkg://MEDS_transforms.configs]",
#     "++do_overwrite=False",
#     "++do_copy=True",
# ]

# result = run_command(cmd, env=base_env)

## Stages step by step

In [9]:
STAGES = [
    "shard_events",
    "split_and_shard_subjects",
    "convert_to_sharded_events",
    "merge_to_MEDS_cohort",
    "extract_code_metadata",
    "finalize_MEDS_metadata",
    "finalize_MEDS_data",
]

STAGES

['shard_events',
 'split_and_shard_subjects',
 'convert_to_sharded_events',
 'merge_to_MEDS_cohort',
 'extract_code_metadata',
 'finalize_MEDS_metadata',
 'finalize_MEDS_data']

### Function to execute a stage

In [10]:
def run_events_direct(name: str):
    cmd = [
        f"MEDS_extract-{name}",
        f"--config-dir={str(ETL_CFG.parent.resolve())}",
        f"--config-name={ETL_CFG.stem}",
        "hydra.searchpath=[pkg://MEDS_transforms.configs]",
        f"stage={name}",
        "++do_overwrite=False",
        "++do_copy=True",
    ]
    return run_command(cmd, env=base_env)

### Sharding stage - OpenICU Extraction

sharding big tables into smaller tables (not for demo, is too small)

In [11]:
result_shard = run_events_direct("shard_events")

RUNNING:
MEDS_extract-shard_events --config-dir=/workspaces/.test/lib/python3.13/site-packages/MIMIC_IV_MEDS/configs --config-name=ETL hydra.searchpath=[pkg://MEDS_transforms.configs] stage=shard_events ++do_overwrite=False ++do_copy=True

RETURN CODE: 1

STDOUT:

[2026-04-01 11:02:18,826][MEDS_transforms.extract.shard_events][INFO] - Running with config:
input_dir: ${oc.env:PRE_MEDS_DIR}
cohort_dir: ${oc.env:MEDS_COHORT_DIR}
_default_description: 'This is a MEDS pipeline ETL. Please set a more detailed description
  at the top of your specific pipeline

  configuration file.'
log_dir: ${stage_cfg.output_dir}/.logs
do_overwrite: false
seed: 1
stages:
- shard_events
- split_and_shard_subjects
- convert_to_sharded_events
- merge_to_MEDS_cohort
- extract_code_metadata
- finalize_MEDS_metadata
- finalize_MEDS_data
stage_configs:
  shard_events:
    row_chunksize: 200000000
    infer_schema_length: 999999999
    data_input_dir: ${input_dir}
  split_and_shard_subjects:
    is_metadata: true


### Check sharding

In [12]:
for p in sorted((MEDS_COHORT_DIR / "shard_events").rglob("*")):
    print(p)

/workspaces/OpenICU.example/output/project/mimiciv-meds-notebook-run/MEDS_cohort/shard_events/.logs
/workspaces/OpenICU.example/output/project/mimiciv-meds-notebook-run/MEDS_cohort/shard_events/.logs/.hydra
/workspaces/OpenICU.example/output/project/mimiciv-meds-notebook-run/MEDS_cohort/shard_events/.logs/.hydra/config.yaml
/workspaces/OpenICU.example/output/project/mimiciv-meds-notebook-run/MEDS_cohort/shard_events/.logs/.hydra/hydra.yaml
/workspaces/OpenICU.example/output/project/mimiciv-meds-notebook-run/MEDS_cohort/shard_events/.logs/.hydra/overrides.yaml
/workspaces/OpenICU.example/output/project/mimiciv-meds-notebook-run/MEDS_cohort/shard_events/.logs/shard_events_0_2026-04-01_11-02-18.log
/workspaces/OpenICU.example/output/project/mimiciv-meds-notebook-run/MEDS_cohort/shard_events/hosp
/workspaces/OpenICU.example/output/project/mimiciv-meds-notebook-run/MEDS_cohort/shard_events/hosp/admissions
/workspaces/OpenICU.example/output/project/mimiciv-meds-notebook-run/MEDS_cohort/shard

### Split and shard subject stage

split subject into training, tuning and held_out, and shared into these groups 

In [13]:
result_split = run_events_direct("split_and_shard_subjects")

RUNNING:
MEDS_extract-split_and_shard_subjects --config-dir=/workspaces/.test/lib/python3.13/site-packages/MIMIC_IV_MEDS/configs --config-name=ETL hydra.searchpath=[pkg://MEDS_transforms.configs] stage=split_and_shard_subjects ++do_overwrite=False ++do_copy=True

RETURN CODE: 0

STDOUT:

[2026-04-01 11:02:19,459][MEDS_transforms.utils][INFO] - Running split_and_shard_subjects with the following configuration:
input_dir: ${oc.env:PRE_MEDS_DIR}
cohort_dir: ${oc.env:MEDS_COHORT_DIR}
_default_description: 'This is a MEDS pipeline ETL. Please set a more detailed description
  at the top of your specific pipeline

  configuration file.'
log_dir: ${stage_cfg.output_dir}/.logs
do_overwrite: false
seed: 1
stages:
- shard_events
- split_and_shard_subjects
- convert_to_sharded_events
- merge_to_MEDS_cohort
- extract_code_metadata
- finalize_MEDS_metadata
- finalize_MEDS_data
stage_configs:
  shard_events:
    row_chunksize: 200000000
    infer_schema_length: 999999999
    data_input_dir: ${input_

# Convert to Sharded events

Trafo to meds format

In [14]:
result_convert = run_events_direct("convert_to_sharded_events")

RUNNING:
MEDS_extract-convert_to_sharded_events --config-dir=/workspaces/.test/lib/python3.13/site-packages/MIMIC_IV_MEDS/configs --config-name=ETL hydra.searchpath=[pkg://MEDS_transforms.configs] stage=convert_to_sharded_events ++do_overwrite=False ++do_copy=True

RETURN CODE: 1

STDOUT:

[2026-04-01 11:02:20,152][MEDS_transforms.utils][INFO] - Running convert_to_sharded_events with the following configuration:
input_dir: ${oc.env:PRE_MEDS_DIR}
cohort_dir: ${oc.env:MEDS_COHORT_DIR}
_default_description: 'This is a MEDS pipeline ETL. Please set a more detailed description
  at the top of your specific pipeline

  configuration file.'
log_dir: ${stage_cfg.output_dir}/.logs
do_overwrite: false
seed: 1
stages:
- shard_events
- split_and_shard_subjects
- convert_to_sharded_events
- merge_to_MEDS_cohort
- extract_code_metadata
- finalize_MEDS_metadata
- finalize_MEDS_data
stage_configs:
  shard_events:
    row_chunksize: 200000000
    infer_schema_length: 999999999
    data_input_dir: ${inp

# Merge to MEDS cohort

Combines the sharded event outputs into single cohort structure organized by split

In [15]:
result_merge = run_events_direct("merge_to_MEDS_cohort")

RUNNING:
MEDS_extract-merge_to_MEDS_cohort --config-dir=/workspaces/.test/lib/python3.13/site-packages/MIMIC_IV_MEDS/configs --config-name=ETL hydra.searchpath=[pkg://MEDS_transforms.configs] stage=merge_to_MEDS_cohort ++do_overwrite=False ++do_copy=True

RETURN CODE: 1

STDOUT:

[2026-04-01 11:02:20,900][MEDS_transforms.utils][INFO] - Running merge_to_MEDS_cohort with the following configuration:
input_dir: ${oc.env:PRE_MEDS_DIR}
cohort_dir: ${oc.env:MEDS_COHORT_DIR}
_default_description: 'This is a MEDS pipeline ETL. Please set a more detailed description
  at the top of your specific pipeline

  configuration file.'
log_dir: ${stage_cfg.output_dir}/.logs
do_overwrite: false
seed: 1
stages:
- shard_events
- split_and_shard_subjects
- convert_to_sharded_events
- merge_to_MEDS_cohort
- extract_code_metadata
- finalize_MEDS_metadata
- finalize_MEDS_data
stage_configs:
  shard_events:
    row_chunksize: 200000000
    infer_schema_length: 999999999
    data_input_dir: ${input_dir}
  split

# Extract code metadata

codes, item-label mappings

In [16]:
result_codes = run_events_direct("extract_code_metadata")

RUNNING:
MEDS_extract-extract_code_metadata --config-dir=/workspaces/.test/lib/python3.13/site-packages/MIMIC_IV_MEDS/configs --config-name=ETL hydra.searchpath=[pkg://MEDS_transforms.configs] stage=extract_code_metadata ++do_overwrite=False ++do_copy=True

RETURN CODE: 1

STDOUT:

[2026-04-01 11:02:21,638][MEDS_transforms.utils][INFO] - Running extract_code_metadata with the following configuration:
input_dir: ${oc.env:PRE_MEDS_DIR}
cohort_dir: ${oc.env:MEDS_COHORT_DIR}
_default_description: 'This is a MEDS pipeline ETL. Please set a more detailed description
  at the top of your specific pipeline

  configuration file.'
log_dir: ${stage_cfg.output_dir}/.logs
do_overwrite: false
seed: 1
stages:
- shard_events
- split_and_shard_subjects
- convert_to_sharded_events
- merge_to_MEDS_cohort
- extract_code_metadata
- finalize_MEDS_metadata
- finalize_MEDS_data
stage_configs:
  shard_events:
    row_chunksize: 200000000
    infer_schema_length: 999999999
    data_input_dir: ${input_dir}
  sp

# Finalize MEDS metadata

unified metadata

In [17]:
result_meta = run_events_direct("finalize_MEDS_metadata")

RUNNING:
MEDS_extract-finalize_MEDS_metadata --config-dir=/workspaces/.test/lib/python3.13/site-packages/MIMIC_IV_MEDS/configs --config-name=ETL hydra.searchpath=[pkg://MEDS_transforms.configs] stage=finalize_MEDS_metadata ++do_overwrite=False ++do_copy=True

RETURN CODE: 0

STDOUT:

[2026-04-01 11:02:22,379][MEDS_transforms.utils][INFO] - Running finalize_MEDS_metadata with the following configuration:
input_dir: ${oc.env:PRE_MEDS_DIR}
cohort_dir: ${oc.env:MEDS_COHORT_DIR}
_default_description: 'This is a MEDS pipeline ETL. Please set a more detailed description
  at the top of your specific pipeline

  configuration file.'
log_dir: ${stage_cfg.output_dir}/.logs
do_overwrite: false
seed: 1
stages:
- shard_events
- split_and_shard_subjects
- convert_to_sharded_events
- merge_to_MEDS_cohort
- extract_code_metadata
- finalize_MEDS_metadata
- finalize_MEDS_data
stage_configs:
  shard_events:
    row_chunksize: 200000000
    infer_schema_length: 999999999
    data_input_dir: ${input_dir}
 

# Finalize MEDS data
final training, tuning, held file

In [18]:
result_data = run_events_direct("finalize_MEDS_data")

RUNNING:
MEDS_extract-finalize_MEDS_data --config-dir=/workspaces/.test/lib/python3.13/site-packages/MIMIC_IV_MEDS/configs --config-name=ETL hydra.searchpath=[pkg://MEDS_transforms.configs] stage=finalize_MEDS_data ++do_overwrite=False ++do_copy=True

RETURN CODE: 1

STDOUT:

[2026-04-01 11:02:23,114][MEDS_transforms.utils][INFO] - Running finalize_MEDS_data with the following configuration:
input_dir: ${oc.env:PRE_MEDS_DIR}
cohort_dir: ${oc.env:MEDS_COHORT_DIR}
_default_description: 'This is a MEDS pipeline ETL. Please set a more detailed description
  at the top of your specific pipeline

  configuration file.'
log_dir: ${stage_cfg.output_dir}/.logs
do_overwrite: false
seed: 1
stages:
- shard_events
- split_and_shard_subjects
- convert_to_sharded_events
- merge_to_MEDS_cohort
- extract_code_metadata
- finalize_MEDS_metadata
- finalize_MEDS_data
stage_configs:
  shard_events:
    row_chunksize: 200000000
    infer_schema_length: 999999999
    data_input_dir: ${input_dir}
  split_and_s